This script:

Reads each tile PNG in the images/ folder

Finds the matching JSON sidecar in meta/ (same tile_id) to retrieve the pixel size (meters per pixel) from the affine transform

Converts 10 cm ground distance into the corresponding grid spacing in pixels

Draws a grid where each line color is camouflaged by computing the average RGB value of neighboring pixels around that line

Saves the new images to a separate output folder (without overwriting the originals unless explicitly configured)

In [1]:
from pathlib import Path
import json
import numpy as np
from PIL import Image

# ---------------- USER SETTINGS ----------------
from pathlib import Path

ROOT = Path("./output/tiles_out_pseudo_ortho")

def select_case(root: Path) -> str:
    cases = sorted([p for p in root.iterdir() if p.is_dir() and not p.name.startswith('.')])

    if not cases:
        raise FileNotFoundError(f"No case folders found in {root}")

    print("\nAvailable cases:")
    for i, c in enumerate(cases, 1):
        print(f"{i}) {c.name}")

    choice = int(input("\nSelect case number: "))
    if choice < 1 or choice > len(cases):
        raise ValueError("Invalid selection")

    return cases[choice - 1].name


CASE = select_case(ROOT)
print("Selected case:", CASE)

TILES_DIR = Path(f"./output/tiles_out_pseudo_ortho/{CASE}/images")
META_DIR  = Path(f"./output/tiles_out_pseudo_ortho/{CASE}/meta")
OUT_DIR   = Path(f"./output/tiles_out_pseudo_ortho/{CASE}/{CASE}_images_grid10cm")

GRID_CM = 10.0              # desired grid size in cm (real-world)
LINE_THICKNESS_PX = 1       # grid line thickness
NEIGHBOR_BAND_PX = 6        # how far to sample neighbors to compute camouflage color
ALPHA = 0.8                 # 1.0 = use computed avg color fully; <1 blends with original pixels

# IMPORTANT for pseudo-ortho: set pixel size (meters per pixel).
# Example: 0.005 = 0.5 cm/pixel, 0.01 = 1.0 cm/pixel
PIXEL_SIZE_M = 0.0075   # 0.75 cm/pixel        # <-- REQUIRED. Set this to your best estimate, e.g. 0.0075
# ------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

def get_pixel_size_m_from_meta(meta_path: Path) -> float:
    """
    Pseudo-ortho tiles (from JPG stitch) typically have NO affine transform.
    We therefore use a user-specified PIXEL_SIZE_M (meters/pixel).
    """
    if PIXEL_SIZE_M is None:
        raise ValueError(
            "PIXEL_SIZE_M is not set. Pseudo-ortho tiles have no meta['transform'].\n"
            "Set PIXEL_SIZE_M in USER SETTINGS, e.g. 0.0075 for 0.75 cm/pixel."
        )
    return float(PIXEL_SIZE_M)

def grid_spacing_px(pixel_size_m: float, grid_cm: float) -> int:
    grid_m = grid_cm / 100.0
    px = int(round(grid_m / pixel_size_m))
    return max(px, 2)  # avoid degenerate spacing

def camo_color_for_vertical_line(img: np.ndarray, x: int, thickness: int, band: int) -> np.ndarray:
    H, W, _ = img.shape
    x0 = max(0, x - thickness // 2)
    x1 = min(W, x0 + thickness)

    left0  = max(0, x0 - band)
    left1  = x0
    right0 = x1
    right1 = min(W, x1 + band)

    samples = []
    if left1 > left0:
        samples.append(img[:, left0:left1, :])
    if right1 > right0:
        samples.append(img[:, right0:right1, :])

    if not samples:
        patch0 = max(0, x - band)
        patch1 = min(W, x + band)
        samples = [img[:, patch0:patch1, :]]

    samp = np.concatenate(samples, axis=1)
    mean_rgb = np.mean(samp.reshape(-1, 3), axis=0)
    return np.clip(np.round(mean_rgb), 0, 255).astype(np.uint8)

def camo_color_for_horizontal_line(img: np.ndarray, y: int, thickness: int, band: int) -> np.ndarray:
    H, W, _ = img.shape
    y0 = max(0, y - thickness // 2)
    y1 = min(H, y0 + thickness)

    top0 = max(0, y0 - band)
    top1 = y0
    bot0 = y1
    bot1 = min(H, y1 + band)

    samples = []
    if top1 > top0:
        samples.append(img[top0:top1, :, :])
    if bot1 > bot0:
        samples.append(img[bot0:bot1, :, :])

    if not samples:
        patch0 = max(0, y - band)
        patch1 = min(H, y + band)
        samples = [img[patch0:patch1, :, :]]

    samp = np.concatenate(samples, axis=0)
    mean_rgb = np.mean(samp.reshape(-1, 3), axis=0)
    return np.clip(np.round(mean_rgb), 0, 255).astype(np.uint8)

def draw_grid_camo(img_u8: np.ndarray, spacing: int, thickness: int, band: int, alpha: float) -> np.ndarray:
    out = img_u8.copy()
    H, W, _ = out.shape

    # Vertical lines
    for x in range(0, W, spacing):
        color = camo_color_for_vertical_line(out, x, thickness, band).astype(np.float32)
        x0 = max(0, x - thickness // 2)
        x1 = min(W, x0 + thickness)
        out[:, x0:x1, :] = np.clip(
            (1 - alpha) * out[:, x0:x1, :].astype(np.float32) + alpha * color,
            0, 255
        ).astype(np.uint8)

    # Horizontal lines
    for y in range(0, H, spacing):
        color = camo_color_for_horizontal_line(out, y, thickness, band).astype(np.float32)
        y0 = max(0, y - thickness // 2)
        y1 = min(H, y0 + thickness)
        out[y0:y1, :, :] = np.clip(
            (1 - alpha) * out[y0:y1, :, :].astype(np.float32) + alpha * color,
            0, 255
        ).astype(np.uint8)

    return out

def process_folder():
    pngs = sorted(TILES_DIR.glob("*.png"))
    if not pngs:
        raise FileNotFoundError(f"No PNG tiles found in: {TILES_DIR.resolve()}")

    done = 0
    skipped = 0

    # spacing is the same for all tiles if PIXEL_SIZE_M is constant
    px_m = get_pixel_size_m_from_meta(Path("dummy.json"))
    spacing = grid_spacing_px(px_m, GRID_CM)
    print(f"Using PIXEL_SIZE_M={px_m} m/px => grid spacing ~ {spacing} px for {GRID_CM} cm")

    for png_path in pngs:
        tile_id = png_path.stem
        meta_path = META_DIR / f"{tile_id}.json"
        if not meta_path.exists():
            print(f"Skip (no meta): {png_path.name}")
            skipped += 1
            continue

        img = Image.open(png_path).convert("RGB")
        img_u8 = np.array(img, dtype=np.uint8)

        out_u8 = draw_grid_camo(
            img_u8,
            spacing=spacing,
            thickness=LINE_THICKNESS_PX,
            band=NEIGHBOR_BAND_PX,
            alpha=ALPHA
        )

        out_path = OUT_DIR / png_path.name
        Image.fromarray(out_u8).save(out_path)
        done += 1

    print("\nFinished.")
    print("Saved:", done)
    print("Skipped:", skipped)
    print("Output folder:", OUT_DIR.resolve())

process_folder()


Available cases:
1) a10_segment1
2) a58_segment2



Select case number:  1


Selected case: a10_segment1
Using PIXEL_SIZE_M=0.0075 m/px => grid spacing ~ 13 px for 10.0 cm

Finished.
Saved: 727
Skipped: 0
Output folder: /home/eghase02/ML4seeding/examples/output/tiles_out_pseudo_ortho/a10_segment1/a10_segment1_images_grid10cm


In [2]:
from pathlib import Path
import json
import numpy as np
from PIL import Image

# ---------------- USER SETTINGS ----------------
from pathlib import Path

ROOT = Path("./output/tiles_out_pseudo_ortho")

def select_case(root: Path) -> str:
    cases = sorted([p for p in root.iterdir() if p.is_dir() and not p.name.startswith('.')])

    if not cases:
        raise FileNotFoundError(f"No case folders found in {root}")

    print("\nAvailable cases:")
    for i, c in enumerate(cases, 1):
        print(f"{i}) {c.name}")

    choice = int(input("\nSelect case number: "))
    if choice < 1 or choice > len(cases):
        raise ValueError("Invalid selection")

    return cases[choice - 1].name


CASE = select_case(ROOT)
print("Selected case:", CASE)

TILES_DIR = Path(f"./output/tiles_out_pseudo_ortho/{CASE}/images")
META_DIR  = Path(f"./output/tiles_out_pseudo_ortho/{CASE}/meta")
OUT_DIR   = Path(f"./output/tiles_out_pseudo_ortho/{CASE}/{CASE}_images_grid10cm")

GRID_CM = 10.0              # desired grid size in cm (real-world)
LINE_THICKNESS_PX = 1       # grid line thickness
NEIGHBOR_BAND_PX = 6        # how far to sample neighbors to compute camouflage color
ALPHA = 0.8                 # 1.0 = use computed avg color fully; <1 blends with original pixels

# IMPORTANT for pseudo-ortho: set pixel size (meters per pixel).
# Example: 0.005 = 0.5 cm/pixel, 0.01 = 1.0 cm/pixel
PIXEL_SIZE_M = 0.0075   # 0.75 cm/pixel        # <-- REQUIRED. Set this to your best estimate, e.g. 0.0075
# ------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

def get_pixel_size_m_from_meta(meta_path: Path) -> float:
    """
    Pseudo-ortho tiles (from JPG stitch) typically have NO affine transform.
    We therefore use a user-specified PIXEL_SIZE_M (meters/pixel).
    """
    if PIXEL_SIZE_M is None:
        raise ValueError(
            "PIXEL_SIZE_M is not set. Pseudo-ortho tiles have no meta['transform'].\n"
            "Set PIXEL_SIZE_M in USER SETTINGS, e.g. 0.0075 for 0.75 cm/pixel."
        )
    return float(PIXEL_SIZE_M)

def grid_spacing_px(pixel_size_m: float, grid_cm: float) -> int:
    grid_m = grid_cm / 100.0
    px = int(round(grid_m / pixel_size_m))
    return max(px, 2)  # avoid degenerate spacing

def camo_color_for_vertical_line(img: np.ndarray, x: int, thickness: int, band: int) -> np.ndarray:
    H, W, _ = img.shape
    x0 = max(0, x - thickness // 2)
    x1 = min(W, x0 + thickness)

    left0  = max(0, x0 - band)
    left1  = x0
    right0 = x1
    right1 = min(W, x1 + band)

    samples = []
    if left1 > left0:
        samples.append(img[:, left0:left1, :])
    if right1 > right0:
        samples.append(img[:, right0:right1, :])

    if not samples:
        patch0 = max(0, x - band)
        patch1 = min(W, x + band)
        samples = [img[:, patch0:patch1, :]]

    samp = np.concatenate(samples, axis=1)
    mean_rgb = np.mean(samp.reshape(-1, 3), axis=0)
    return np.clip(np.round(mean_rgb), 0, 255).astype(np.uint8)

def camo_color_for_horizontal_line(img: np.ndarray, y: int, thickness: int, band: int) -> np.ndarray:
    H, W, _ = img.shape
    y0 = max(0, y - thickness // 2)
    y1 = min(H, y0 + thickness)

    top0 = max(0, y0 - band)
    top1 = y0
    bot0 = y1
    bot1 = min(H, y1 + band)

    samples = []
    if top1 > top0:
        samples.append(img[top0:top1, :, :])
    if bot1 > bot0:
        samples.append(img[bot0:bot1, :, :])

    if not samples:
        patch0 = max(0, y - band)
        patch1 = min(H, y + band)
        samples = [img[patch0:patch1, :, :]]

    samp = np.concatenate(samples, axis=0)
    mean_rgb = np.mean(samp.reshape(-1, 3), axis=0)
    return np.clip(np.round(mean_rgb), 0, 255).astype(np.uint8)

def draw_grid_camo(img_u8: np.ndarray, spacing: int, thickness: int, band: int, alpha: float) -> np.ndarray:
    out = img_u8.copy()
    H, W, _ = out.shape

    # Vertical lines
    for x in range(0, W, spacing):
        color = camo_color_for_vertical_line(out, x, thickness, band).astype(np.float32)
        x0 = max(0, x - thickness // 2)
        x1 = min(W, x0 + thickness)
        out[:, x0:x1, :] = np.clip(
            (1 - alpha) * out[:, x0:x1, :].astype(np.float32) + alpha * color,
            0, 255
        ).astype(np.uint8)

    # Horizontal lines
    for y in range(0, H, spacing):
        color = camo_color_for_horizontal_line(out, y, thickness, band).astype(np.float32)
        y0 = max(0, y - thickness // 2)
        y1 = min(H, y0 + thickness)
        out[y0:y1, :, :] = np.clip(
            (1 - alpha) * out[y0:y1, :, :].astype(np.float32) + alpha * color,
            0, 255
        ).astype(np.uint8)

    return out

def process_folder():
    pngs = sorted(TILES_DIR.glob("*.png"))
    if not pngs:
        raise FileNotFoundError(f"No PNG tiles found in: {TILES_DIR.resolve()}")

    done = 0
    skipped = 0

    # spacing is the same for all tiles if PIXEL_SIZE_M is constant
    px_m = get_pixel_size_m_from_meta(Path("dummy.json"))
    spacing = grid_spacing_px(px_m, GRID_CM)
    print(f"Using PIXEL_SIZE_M={px_m} m/px => grid spacing ~ {spacing} px for {GRID_CM} cm")

    for png_path in pngs:
        tile_id = png_path.stem
        meta_path = META_DIR / f"{tile_id}.json"
        if not meta_path.exists():
            print(f"Skip (no meta): {png_path.name}")
            skipped += 1
            continue

        img = Image.open(png_path).convert("RGB")
        img_u8 = np.array(img, dtype=np.uint8)

        out_u8 = draw_grid_camo(
            img_u8,
            spacing=spacing,
            thickness=LINE_THICKNESS_PX,
            band=NEIGHBOR_BAND_PX,
            alpha=ALPHA
        )

        out_path = OUT_DIR / png_path.name
        Image.fromarray(out_u8).save(out_path)
        done += 1

    print("\nFinished.")
    print("Saved:", done)
    print("Skipped:", skipped)
    print("Output folder:", OUT_DIR.resolve())

process_folder()


Available cases:
1) a10_segment1
2) a58_segment2



Select case number:  2


Selected case: a58_segment2
Using PIXEL_SIZE_M=0.0075 m/px => grid spacing ~ 13 px for 10.0 cm

Finished.
Saved: 1415
Skipped: 0
Output folder: /home/eghase02/ML4seeding/examples/output/tiles_out_pseudo_ortho/a58_segment2/a58_segment2_images_grid10cm
